# Claim-guided semantic seeds — Colab pilot
**Prepared, not executed. Awaiting your authorization.** No local computer is required: upload this notebook to Google Colab and choose a GPU runtime after authorization.

Human-written claim → LLM-selected semantic spans → equal seed mass per span → existing graph relevance propagation → pruning.
No SHAP. Agent-generated claims remain deferred.

This notebook carries an embedded snapshot of the current implementation changes, applied to a pinned repository commit. They have not yet passed runtime tests; the test cell must pass before experimentation.

The small pilot tests feasibility, not a general mechanistic conclusion. Semantic relevance is a search priority, not measured causal importance. Interventions below change internal feature activations to measure effects on answers.


In [ ]:
# Leave False until you authorize execution. This notebook has not been run.
AUTHORIZED = False
SPANS_REVIEWED = False
RUN_INTERVENTIONS = False

def require_authorization():
    if not AUTHORIZED:
        raise RuntimeError("Waiting for your authorization. No setup, API call, or experiment has run.")

SEED = 42
MODEL_NAME = "google/gemma-2-2b"
TRANSCODER = "mntss/clt-gemma-2-2b-2.5M"
SELECTOR_MODEL = "gemini-2.5-flash"
CLAIM = "The model combines the named country with the capital relation to retrieve and promote the capital answer."
MAX_FEATURE_NODES = 2048
ATTRIBUTION_BATCH_SIZE = 32
NODE_THRESHOLDS = [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
# Development first. Enable held-out evaluation only after fixing all settings.
SPLIT = "development"


## 1. Prepare the pinned source and install dependencies
Setup uses a dedicated directory in the Colab runtime. It does not publish changes or modify your GitHub repository. Installation and test logs are saved with bounded output.


In [ ]:
require_authorization()
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone

REPO = Path("/content/circuit_semantic_pilot")
SETUP_LOG = Path("/content/semantic_setup.log")

def run_logged(command, cwd=None, log=SETUP_LOG):
    with log.open("a") as stream:
        result = subprocess.run(command, cwd=cwd, stdout=stream, stderr=subprocess.STDOUT)
    print(log.read_text(errors="replace")[-4000:])
    if result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}); see {log}.")


In [ ]:
require_authorization()
BASE_COMMIT = '6a15250e4ea442c8bceaa8e3090c1caaa76a1bca'
SOURCE_SHA256 = '66cb3add047407d0ccd8e1f9b63b25c580ab96650a9fbc8c9f4e476a5cc3c132'
SOURCE_ZIP_BASE64 = '...'
DELETED_PATHS = ['summarization/token_attribution.py', 'tests/test_token_attribution_defaults.py']
if not REPO.exists():
    run_logged(["git", "clone", "https://github.com/IamKrill1n/circuit_tracer_mod.git", str(REPO)])
    run_logged(["git", "checkout", "--detach", BASE_COMMIT], cwd=REPO)
current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if current != BASE_COMMIT:
    raise RuntimeError("Existing Colab checkout does not match the pinned commit. Use a fresh runtime.")
source_bytes = base64.b64decode(SOURCE_ZIP_BASE64)
assert hashlib.sha256(source_bytes).hexdigest() == SOURCE_SHA256
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    for member in archive.infolist():
        target = (REPO / member.filename).resolve()
        if not target.is_relative_to(REPO.resolve()):
            raise ValueError("Invalid source snapshot path")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
for relative in DELETED_PATHS:
    (REPO / relative).unlink(missing_ok=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
run_logged([sys.executable, "-m", "pip", "install", "-e", ".[dev]", "google-genai", "python-dotenv", "plotly"])
print("Source prepared:", BASE_COMMIT, SOURCE_SHA256)


## 2. Verify the implementation before using credentials
These tests use synthetic graphs and mocked LLM responses. Failures stop the workflow; save the log and send it back for correction. No paid API calls belong in this step.


In [ ]:
require_authorization()
run_logged([sys.executable, "-m", "pytest", "-v", "tests/test_semantic_seeds.py",
            "tests/test_pipeline_cli.py", "tests/test_visualization_app.py", "tests/test_prune.py",
            "-m", "not requires_disk", "-x"], cwd=REPO, log=Path("/content/semantic_tests.log"))
TESTS_PASSED = True


## 3. Credentials, GPU, and reproducibility
Add `HF_TOKEN` and `GOOGLE_API_KEY` to Colab Secrets and grant notebook access. The Hugging Face account must have access to the chosen model. For an OpenAI selector, use a registry model and its `OPENAI_API_KEY` instead.
Only the human claim, synthetic prompt, and indexed tokens go to the selector provider. Credentials are read into memory, never printed or saved in results.


In [ ]:
require_authorization()
assert TESTS_PASSED
from google.colab import userdata
registry = json.loads((REPO / "summarization/llm_models.json").read_text())
selector_key = registry[SELECTOR_MODEL]["api_key_env"]
os.environ[selector_key] = userdata.get(selector_key)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGINGFACE_API_KEY"] = os.environ["HF_TOKEN"]
import random
import numpy as np
import torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select a GPU runtime in Colab before continuing."
torch.cuda.manual_seed_all(SEED)
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
RUN_DIR = REPO / "experiments/runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
config = dict(seed=SEED, claim=CLAIM, model=MODEL_NAME, transcoder=TRANSCODER,
              selector=SELECTOR_MODEL, git_commit=BASE_COMMIT, source_sha256=SOURCE_SHA256,
              split=SPLIT, node_thresholds=NODE_THRESHOLDS, max_feature_nodes=MAX_FEATURE_NODES,
              attribution_batch_size=ATTRIBUTION_BATCH_SIZE, dtype=str(DTYPE),
              gpu=torch.cuda.get_device_name(), attention_frozen=False)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
with (RUN_DIR / "packages.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=stream, check=True)
print("GPU:", config["gpu"], "Results:", RUN_DIR)


## 4. Fixed factual pilot and behavior check
Development countries: France, Germany. Held-out countries: Italy, Spain. Each gets a direct prompt, a paraphrase, and a distractor version. Expected spans are human-authored entity and relation annotations; they are not supplied to the selector.

Only correctly answered, single-token capital cases enter circuit comparison. Incorrect answers and unsupported tokenizations remain in `behavior.json`. This tiny set is for feasibility; freeze choices before switching to the held-out split.


In [ ]:
require_authorization()
from circuit_tracer import ReplacementModel, attribute
from summarization.attr_graph import AttrGraph
from summarization.semantic_seeds import select_semantic_seeds
from summarization.prune import prune_attr_graph, save_prune_graph
from eval.prune_graphs import seed_conditions

country_sets = {"development": [("France", "Paris"), ("Germany", "Berlin")],
                "held_out": [("Italy", "Rome"), ("Spain", "Madrid")]}
examples = []
for country, answer in country_sets[SPLIT]:
    distractor = "Italy" if SPLIT == "development" else "France"
    prompts = {
        "direct": f"The capital of {country} is",
        "paraphrase": f"The country is {country}. Its capital city is",
        "distractor": f"Travel note: {distractor}. The capital of {country} is",
    }
    for variant, prompt in prompts.items():
        examples.append(dict(id=f"{country.lower()}_{variant}", country=country,
                             answer=answer, prompt=prompt, variant=variant, split=SPLIT,
                             expected_terms=[country, "capital city" if variant == "paraphrase" else "capital"]))
(RUN_DIR / "examples.json").write_text(json.dumps(examples, indent=2))
model = ReplacementModel.from_pretrained(MODEL_NAME, TRANSCODER, dtype=DTYPE,
                                        lazy_encoder=True, backend="transformerlens", device="cuda")
behavior = []
for example in examples:
    prefix_ids = model.tokenizer.encode(example["prompt"], add_special_tokens=False)
    completed_ids = model.tokenizer.encode(example["prompt"] + " " + example["answer"], add_special_tokens=False)
    answer_ids = completed_ids[len(prefix_ids):] if completed_ids[:len(prefix_ids)] == prefix_ids else []
    with torch.inference_mode():
        logits = model(example["prompt"]).reshape(-1, model.cfg.d_vocab)[-1]
    predicted_id = int(logits.argmax())
    eligible = len(answer_ids) == 1 and predicted_id == answer_ids[0]
    behavior.append({**example, "answer_ids": answer_ids, "prediction": model.tokenizer.decode([predicted_id]),
                     "eligible": eligible, "status": "correct" if eligible else "incorrect_or_multitoken"})
(RUN_DIR / "behavior.json").write_text(json.dumps(behavior, indent=2))
print([{k: row[k] for k in ("id", "prediction", "status")} for row in behavior])


## 5. Trace and review semantic spans
The selector sees the exact prompt and graph token positions. The table shows human-expected positions beside the selected positions. Matching positions measures semantic agreement, not causality.

Inspect wrong selections, distractors, split words, and paraphrase consistency. **Stop here for review.** Change `SPANS_REVIEWED` only after accepting the selections; do not tune the selector on held-out examples.


In [ ]:
require_authorization()
import pandas as pd

def term_positions(ag, term):
    tokens = ag.metadata["prompt_tokens"]
    text = "".join(tokens)
    if text.count(term) != 1:
        raise ValueError(f"Expected one occurrence of human-annotated term: {term!r}")
    start = text.index(term)
    end = start + len(term)
    indices, offset = [], 0
    for i, token in enumerate(tokens):
        if offset < end and offset + len(token) > start:
            indices.append(i)
        offset += len(token)
    return indices

graphs, selections, audit_rows = {}, {}, []
for row in behavior:
    if not row["eligible"]:
        continue
    with torch.inference_mode():
        graph = attribute(prompt=row["prompt"], model=model,
                          attribution_targets=torch.tensor(row["answer_ids"]),
                          max_feature_nodes=MAX_FEATURE_NODES, batch_size=ATTRIBUTION_BATCH_SIZE,
                          offload="cpu", verbose=False)
    graph.to_pt(str(RUN_DIR / f"{row['id']}.pt"))
    ag = AttrGraph.from_graph(graph)
    ag.adj = ag.adj.cpu()
    graphs[row["id"]] = ag
    weights = select_semantic_seeds(ag, CLAIM, SELECTOR_MODEL)
    selections[row["id"]] = weights
    expected = set(i for term in row["expected_terms"] for i in term_positions(ag, term))
    selected = set(i for span in ag.metadata["semantic_seed"]["spans"] for i in range(span["start"], span["end"]))
    audit_rows.append(dict(id=row["id"], expected_positions=sorted(expected), selected_positions=sorted(selected),
                           token_overlap=len(expected & selected) / len(expected | selected),
                           spans=ag.metadata["semantic_seed"]["spans"]))
    (RUN_DIR / f"{row['id']}_selection.json").write_text(json.dumps(ag.metadata, indent=2))
    del graph
    torch.cuda.empty_cache()
(RUN_DIR / "span_audit.json").write_text(json.dumps(audit_rows, indent=2))
display(pd.DataFrame(audit_rows))


## 6. Prune with four seed conditions
Compare semantic spans, uniform ordinary tokens, shuffled semantic weights, and output influence alone. The shuffle has an explicit seed and leaves special tokens at zero. Graph propagation is unchanged.

Thresholds do not guarantee equal graph sizes. Record actual feature counts and restrict paired comparisons to **exact shared counts**. No common count means the pilot has no matched-size comparison; report that rather than treating equal thresholds as equal budgets. The full-size endpoint is a sanity check, not evidence of better compression.


In [ ]:
require_authorization()
assert SPANS_REVIEWED, "Pause for span review before pruning."
pruned, records = {}, []
for example_id, ag in graphs.items():
    for condition, weights in seed_conditions(ag, selections[example_id], SEED).items():
        for threshold in NODE_THRESHOLDS:
            pg = prune_attr_graph(ag, token_weights=weights, node_threshold=threshold,
                                 edge_threshold=0.95, keep_all_tokens_and_logits=True,
                                 combine_method="arithmetic" if condition == "output_only" else "geometric",
                                 alpha=1.0 if condition == "output_only" else 0.5)
            key = f"{example_id}_{condition}_{threshold}"
            path = RUN_DIR / "pruned" / f"{key}.pt"
            path.parent.mkdir(exist_ok=True)
            pg.metadata = {**pg.metadata, "seed_condition": condition, "seed": SEED}
            save_prune_graph(pg, str(path))
            pruned[key] = pg
            records.append(dict(key=key, id=example_id, condition=condition, threshold=threshold,
                                num_features=sum(n.feature_type == "cross layer transcoder" for n in pg.nodes),
                                num_nodes=pg.num_nodes, num_edges=pg.num_edges))
(RUN_DIR / "pruning.json").write_text(json.dumps(records, indent=2))
display(pd.DataFrame(records).pivot_table(index=["id", "threshold"], columns="condition", values="num_features"))


## 7. Optional donor-state intervention evaluation
Set `RUN_INTERVENTIONS=True` only after reviewing the earlier results. This step can be expensive.

For each eligible country pair with the same prompt variant, replace source-country feature activations with donor-country values. Source and donor subject positions come from the **human entity annotation**, not the LLM-selected spans. Unequal subject-token counts are reported as unsupported instead of blindly aligning positions.

Compare the full candidate graph with each retained graph. Outside-retained *candidate* features are zeroed; features absent from the original graph remain untouched. Thus this tests pruning within a candidate graph, not a complete isolated circuit. Attention is unfrozen in these interventions. Edges are not intervened on, so this does not validate edge pruning or prove the whole retrieval mechanism.

Measure both answer probabilities and the donor-minus-source score margin. Compare the patch effect relative to each graph's own unpatched baseline; preserving the answer alone is insufficient.


In [ ]:
require_authorization()
assert SPANS_REVIEWED and RUN_INTERVENTIONS, "Intervention execution is not enabled."

def feature_key(node):
    layer, feature, pos = map(int, node.node_id.split("_"))
    return layer, pos, feature

def measure(prompt, interventions, source_id, donor_id):
    with torch.inference_mode():
        logits, _ = model.feature_intervention(prompt, interventions, freeze_attention=False, return_activations=False)
    last = logits.reshape(-1, logits.shape[-1])[-1].float()
    probs = last.softmax(-1)
    return dict(margin=float(last[donor_id] - last[source_id]),
                source_probability=float(probs[source_id]), donor_probability=float(probs[donor_id]))

intervention_rows, skipped_pairs = [], []
eligible = {row["id"]: row for row in behavior if row["id"] in graphs}
for source in eligible.values():
    donor = next((r for r in eligible.values() if r["country"] != source["country"] and r["variant"] == source["variant"]), None)
    if donor is None:
        skipped_pairs.append({"source": source["id"], "reason": "No eligible donor"})
        continue
    ag, donor_ag = graphs[source["id"]], graphs[donor["id"]]
    source_pos = term_positions(ag, source["country"])
    donor_pos = term_positions(donor_ag, donor["country"])
    if len(source_pos) != len(donor_pos):
        skipped_pairs.append({"source": source["id"], "donor": donor["id"], "reason": "Unequal entity-token counts"})
        continue
    alignment = dict(zip(source_pos, donor_pos))
    with torch.inference_mode():
        _, donor_acts = model.get_activations(donor["prompt"], sparse=False)
    donor_acts = donor_acts.cpu()  # (layers, positions, features)
    all_features = {feature_key(n) for n in ag.nodes if n.feature_type == "cross layer transcoder"}
    patch = {key: float(donor_acts[key[0], alignment[key[1]], key[2]])
             for key in all_features if key[1] in alignment}
    del donor_acts
    if not patch:
        skipped_pairs.append({"source": source["id"], "reason": "No candidate features at subject positions"})
        continue
    src_id, dst_id = source["answer_ids"][0], donor["answer_ids"][0]
    full_base = measure(source["prompt"], [], src_id, dst_id)
    full_patch = measure(source["prompt"], [(*k, v) for k, v in patch.items()], src_id, dst_id)
    full_effect = full_patch["margin"] - full_base["margin"]
    for record in records:
        if record["id"] != source["id"]:
            continue
        pg = pruned[record["key"]]
        kept = {feature_key(n) for n in pg.nodes if n.feature_type == "cross layer transcoder"}
        zeros = [(*key, 0.0) for key in sorted(all_features - kept)]
        base = measure(source["prompt"], zeros, src_id, dst_id)
        altered = measure(source["prompt"], zeros + [(*k, v) for k, v in patch.items() if k in kept], src_id, dst_id)
        effect = altered["margin"] - base["margin"]
        intervention_rows.append({**record, "donor": donor["id"], "full_baseline": full_base,
                                  "full_patched": full_patch, "retained_baseline": base,
                                  "retained_patched": altered, "full_effect": full_effect,
                                  "retained_effect": effect, "effect_absolute_error": abs(effect-full_effect),
                                  "margin_absolute_error": abs(base["margin"]-full_base["margin"]),
                                  "full_patch_feature_count": len(patch),
                                  "retained_patch_feature_count": len(set(patch) & kept)})
        (RUN_DIR / "interventions.json").write_text(json.dumps(intervention_rows, indent=2))
    torch.cuda.empty_cache()
(RUN_DIR / "skipped_pairs.json").write_text(json.dumps(skipped_pairs, indent=2))
print("Intervention records:", len(intervention_rows), "Skipped pairs:", len(skipped_pairs))


## 8. Matched-size results and download
Keep every raw result. The table includes only sizes reached by all four conditions for the same source example; duplicate sizes use the lowest predeclared threshold, without selecting by outcomes.

Success would mean better intervention-effect preservation at a compressed, matched size on held-out cases. A zero full-graph effect provides no evidence for the proposed patch mechanism; inspect it explicitly. Do not interpret span agreement or graph appearance as mechanism validation.


In [ ]:
require_authorization()
assert RUN_INTERVENTIONS
results = pd.DataFrame(intervention_rows)
if not results.empty:
    unique = results.sort_values("threshold").drop_duplicates(["id", "condition", "num_features"])
    counts = unique.groupby(["id", "num_features"])["condition"].transform("nunique")
    matched = unique[counts == 4]
    display(matched[["id", "condition", "num_features", "full_effect", "retained_effect",
                     "effect_absolute_error", "margin_absolute_error"]])
    matched.to_json(RUN_DIR / "matched_results.json", orient="records", indent=2)
    if matched.empty:
        print("No exact matched-size comparison. Report as inconclusive; refine thresholds on development data only.")
else:
    print("No supported intervention pairs; inspect behavior.json and skipped_pairs.json.")


In [ ]:
require_authorization()
# Download at any checkpoint, even if later GPU steps fail.
import shutil
from google.colab import files
archive_path = shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
files.download(archive_path)


## Run record
- Status: **not executed; awaiting authorization**.
- Notebook code cells were checked for Python syntax only during preparation.
- The embedded implementation snapshot still needs its focused tests in Colab.
- No observed model outputs, experimental results, or mechanistic conclusions are supplied.
- Download results before ending the runtime; this notebook does not mount or write to Google Drive automatically.
